## 1. Detect zipped or automatically unzipped Kaggle input


In [ ]:
from pathlib import Path
import base64, json, os, subprocess, sys, time, zipfile
INPUT_ROOT = Path(os.environ.get("JCAM_INPUT_ROOT", "/kaggle/input"))
WORK_ROOT = Path(os.environ.get("JCAM_WORK_ROOT", "/kaggle/working"))
RUN_ROOT = WORK_ROOT / "jcam_thm12_bank_coverage"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
RUN_OK = True
STAGES = []
def record(stage, outcome, detail):
    STAGES.append({"stage": stage, "outcome": outcome, "detail": detail})
    print(stage, outcome, detail)
mode = "embedded_processed_source_with_verified_UCI_hash"
if INPUT_ROOT.exists():
    if any(INPUT_ROOT.rglob("bank_marketing_uci.zip")):
        mode = "kaggle_zip_detected; raw source will be verified"
    elif any(INPUT_ROOT.rglob("bank-additional-full.csv")):
        mode = "kaggle_auto_unzipped_detected; embedded processed source selected"
    elif any(INPUT_ROOT.rglob("*.zip")):
        mode = "other_kaggle_zip_detected; embedded processed source selected"
record("bootstrap", "passed", mode)


## 2. Restore dataset, proof, code, and draft


In [ ]:
SOURCE_ARCHIVE = base64.b64decode("".join([
    '...'
]))
(RUN_ROOT / "source_package.zip").write_bytes(SOURCE_ARCHIVE)
with zipfile.ZipFile(RUN_ROOT / "source_package.zip") as archive:
    for member in archive.infolist():
        target = (RUN_ROOT / member.filename).resolve()
        if not target.is_relative_to(RUN_ROOT.resolve()):
            raise RuntimeError(f"Unsafe ZIP member: {member.filename}")
    archive.extractall(RUN_ROOT)
record("sources", "passed", "UCI data, THM-12/APP-02 proofs, experiment, draft and audit")


## 3. Run exact finite-pool Pareto coverage experiment


In [ ]:
started = time.perf_counter()
script = RUN_ROOT / "Code" / "experiments" / "real_bank_pareto_coverage.py"
data = RUN_ROOT / "Code" / "data" / "bank_marketing" / "bank_stream.npz"
raw = next(INPUT_ROOT.rglob("bank_marketing_uci.zip"), None) if INPUT_ROOT.exists() else None
output = RUN_ROOT / "Code" / "results" / "real_bank_pareto_coverage"
cmd = [sys.executable, str(script), "--data", str(data), "--output", str(output)]
if raw is not None:
    cmd.extend(["--raw", str(raw)])
result = subprocess.run(cmd, cwd=RUN_ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(result.stdout[-6000:])
if result.returncode:
    RUN_OK = False
    record("bank_coverage", "failed", f"exit={result.returncode}")
else:
    summary = json.loads((output / "bank_pareto_coverage_summary.json").read_text())
    RUN_OK = bool(summary["chronological_batches"] == 128 and summary["containment_checks"] == 1920 and summary["oracle_checks"] == 1920 and len(summary["by_fraction"]) == 3)
    record("bank_coverage", "passed" if RUN_OK else "failed", f"checks={summary['containment_checks']}; ratios={[round(r['adaptive_to_blind_ratio'], 4) for r in summary['by_fraction']]}; seconds={time.perf_counter()-started:.2f}")


## 4. Zip outputs, exact data, and manuscript sources


In [ ]:
(RUN_ROOT / "notebook_stages.json").write_text(json.dumps({"run_ok": RUN_OK, "stages": STAGES}, indent=2), encoding="utf-8")
FINAL_ZIP = WORK_ROOT / "result_jcam_thm12_bank_coverage.zip"
with zipfile.ZipFile(FINAL_ZIP, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for path in sorted(RUN_ROOT.rglob("*")):
        if path.is_file() and path.name != "source_package.zip":
            archive.write(path, arcname=(Path("thm12_bank_coverage") / path.relative_to(RUN_ROOT)).as_posix())
print("Output archive:", FINAL_ZIP, "bytes:", FINAL_ZIP.stat().st_size, "run_ok:", RUN_OK)
